# Zero-Shot Fault Detection — Colab Runner
**Run cells top to bottom.** Runtime → Change runtime type → **T4 GPU** before starting.

Two dataset paths:
- **Option A (recommended under deadline):** DCASE 2020 Task 2 *development set* — this is the official single-channel, 16 kHz repackaging of **MIMII** (fan/pump/slider/valve) and **ToyADMOS** (ToyCar), ~1–2 GB per machine instead of ~10+ GB. Same recordings, same Zenodo provenance.
- **Option B:** raw MIMII 6 dB archives (huge; only if you have hours to spare).

If you use Option A, change one sentence in the report's *Datasets* section to: *"We use the DCASE 2020 Task 2 development set, which repackages the MIMII and ToyADMOS recordings as single-channel 16 kHz clips."*

In [ ]:
# Cell 1 — GPU check
!nvidia-smi -L
import torch; print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Cell 2 — Get the code. EITHER (a) clone your repo:
# !git clone https://github.com/<you>/zero-shot-fault-detection.git /content/zsfd
# OR (b) upload zero_shot_fault_detection.zip when prompted:
from google.colab import files
up = files.upload()  # pick zero_shot_fault_detection.zip
!unzip -q zero_shot_fault_detection.zip -d /content/
%cd /content/zsfd
!ls src

In [ ]:
# Cell 3 — Dependencies (torch/torchaudio are preinstalled on Colab)
!pip install -q scikit-learn pandas matplotlib tabulate

In [ ]:
# Cell 4 — OPTION A: DCASE 2020 Task 2 dev set (MIMII + ToyADMOS, repackaged)
%cd /content/zsfd
!mkdir -p data/raw
for f in ["dev_data_fan.zip","dev_data_pump.zip","dev_data_slider.zip",
          "dev_data_valve.zip","dev_data_ToyCar.zip"]:
    !wget -q --show-progress "https://zenodo.org/records/3678171/files/{f}?download=1" -O data/raw/{f}
!ls -lh data/raw

In [ ]:
# Cell 5 — Extract and arrange
%cd /content/zsfd
!mkdir -p data/mimii data/toy
for f in ["dev_data_fan","dev_data_pump","dev_data_slider","dev_data_valve"]:
    !unzip -q data/raw/{f}.zip -d data/mimii/
!unzip -q data/raw/dev_data_ToyCar.zip -d data/toy/
# sanity scan
import sys; sys.path.insert(0, "/content/zsfd")
from pathlib import Path
from src.data import list_wavs
for m in ["fan","pump","slider","valve"]:
    n,a = list_wavs(Path("data/mimii"), m); print(f"{m:7s} normal={len(n):5d} abnormal={len(a):4d}")
n,a = list_wavs(Path("data/toy"), "ToyCar"); print(f"ToyCar  normal={len(n):5d} abnormal={len(a):4d}")
# all rows must be non-zero before continuing

### Option B (only if required to use raw archives)
```python
# Raw MIMII 6 dB (~10 GB per machine!):
# for f in ["6_dB_fan.zip","6_dB_pump.zip","6_dB_slider.zip","6_dB_valve.zip"]:
#     !wget -q "https://zenodo.org/records/3384388/files/{f}?download=1" -O data/raw/{f}
#     !unzip -q data/raw/{f} -d data/mimii/
# Raw ToyADMOS is multi-part 7z (needs p7zip):
# !apt-get -qq install p7zip-full
# !wget -q "https://zenodo.org/records/3351307/files/ToyCar.7z.001?download=1" -O data/raw/ToyCar.7z.001
# (download ALL .7z.00x parts, then:)  !7z x data/raw/ToyCar.7z.001 -odata/toy/
```

In [ ]:
# Cell 6 — Smoke run (2 epochs, one fold) to confirm the pipeline end-to-end (~3 min)
%cd /content/zsfd
!python -m src.train --mimii-root data/mimii --source-machines fan pump slider \
    --epochs 2 --out runs/smoke
!python -m src.evaluate --model runs/smoke/model.pt --mimii-root data/mimii \
    --target-machine valve --results-csv results/smoke.csv

In [ ]:
# Cell 7 — FULL ablation grid + domain-gap sweep (~2–3 h at 8 epochs on T4)
# If very short on time: --epochs 5 still gives stable rankings.
%cd /content/zsfd
!rm -f results/results.csv
!python -m src.run_ablations --mimii-root data/mimii --toy-root data/toy --epochs 8

In [ ]:
# Cell 8 — Report tables + degradation plot
%cd /content/zsfd
!python -m src.make_report_tables --results results/results.csv
print(open("results/tables.md").read())
from IPython.display import Image, display
display(Image("results/domain_gap.png"))

In [ ]:
# Cell 9 — Package everything for submission
%cd /content/zsfd
!zip -rq /content/submission_results.zip results runs/*/config.json
from google.colab import files
files.download("/content/submission_results.zip")
# Then: paste tables into the report's [FILL AFTER RUNS] blocks,
# regenerate the PDF locally (python report/make_report.py), push code to GitHub, submit the form.

### If you run out of time
Submit whatever folds completed — `results/results.csv` is appended after **every** fold, so partial results are always available. State in the report which folds ran and which didn't. Real partial numbers beat complete invented ones at the interview.